# 12 — Unexpected credit contraction (flexible FX, leverage 1/16)

**Ports**: `Code/MATLAB/MIT_transition.m` lines 127–157 (“Unexpected credit contraction”) plus the leverage block in `Main.m` lines 342–356. Reuses the `solve_transition_bis.m` and `backsolve_egm.m` ports from notebook 10, but parameterised to take the contraction-specific initial conditions and the leverage parameter `k_const_hat = 1/16`.

**What we compute**: the perfect-foresight transition path triggered by an *unexpected* contraction in foreign credit at quarter 41 of the expansion path (notebook 10). Households were marching along the credit-expansion path when the contraction hits and then re-optimise given the new shock path `new_shock_b_agg` from notebook 03 (which decays back toward the calibrated SS NFA $\bar B \approx -4.34$).

**Initial conditions** (from period 41 of the expansion in notebook 10):
* $k_\text{initial} = k_\text{agg}[40] \approx 29.75$
* $q_\text{initial} = q_\text{agg}[39] \approx 1.017$ (one period earlier, because q is forward-looking)
* $\text{tot}_\text{initial} = \text{tot}[39] \approx 1.026$ (similar reasoning)
* $\text{BG\_shock} = \text{shock\_b\_agg}[39] \approx -6.28$ (households' inherited foreign liabilities)
* $G_0^\text{shock} = \text{dist\_tran}[\cdot,\cdot,40]$ from notebook 10

**Terminal conditions**:
* $k_\text{final} = k_\text{ss}$ from the calibrated SS
* `cpol_calibrated`, `c_fine_calibrated` from notebook 07

**Leverage** (`Main.m` line 345): $\hat k = 1/16$. With $\text{BG\_shock} \ne 0$ the `backsolve_egm` revaluation of gross liabilities is now non-trivial, and a fixed share of capital ($k_\text{const} = \hat k\,k_\text{initial}$) is held uniformly across households.

**Strategy**:
1. Build the simple geometric-decay initial guess from `MIT_transition.m` lines 134–141.
2. Run `scipy.optimize.fsolve` from this guess (no MATLAB warm-start exists for the contraction).
3. Save aggregates for notebook 14 (Figs 4–5).

**Runtime**: ~5–7 minutes single-core (one `fsolve` over 399 unknowns, ~8 Newton iterations × ~0.4s per residual evaluation). Logged precisely with `time.perf_counter()` for transparent reporting.

## Imports + load Phase B/C output

In [1]:
from pathlib import Path
import time

import numpy as np
from scipy.optimize import brentq, fsolve
from scipy.interpolate import PchipInterpolator

OUTPUT_DIR = Path('..') / 'output'

p = np.load(OUTPUT_DIR / 'params.npz')
cb = np.load(OUTPUT_DIR / 'calibration.npz')
sh = np.load(OUTPUT_DIR / 'shock_path.npz')
ini = np.load(OUTPUT_DIR / 'initial_ss.npz')
tr = np.load(OUTPUT_DIR / 'transition_flex.npz')

Agrid = p['params__Agrid']
Agrid_fine = p['params__Agrid_fine']
piex = p['params__piex']
ex = p['params__ex']
nA = int(p['params__nA'])
nA_fine = int(p['params__nA_fine'])
ns = int(p['params__ns'])
Index_b_min = int(p['params__Index_b_min'])
bmin = float(p['params__bmin'])
delta = float(p['params__delta'])
alpha = float(p['params__alpha'])
gamma = float(p['params__gamma'])
theta = float(p['params__theta'])
omega = float(p['params__omega'])
varphi = float(p['params__varphi'])
chi_dis = float(p['params__chi_dis'])
epsilon_f = float(p['params__epsilon_f'])
epsilon_w = float(p['params__epsilon_w'])
z_ss = float(p['params__z_ss'])
ex_mean = float(p['params__ex_mean'])
k_ss = float(p['params__k_ss'])
b_agg_ss = float(p['params__b_agg_ss'])

beta_calibrated = float(cb['beta_calibrated'])
cpol_calibrated = cb['cpol']
c_fine_calibrated = cb['c_fine']

shock_b_agg = sh['shock_b_agg']
new_shock_b_agg = sh['new_shock_b_agg']
TT = int(sh['TT'])
d_s = float(ini['d_s'])
theta_star = int(ini['theta_star'])
z_t = np.ones(TT) * z_ss

# Period-41 state from the expansion (notebook 10)
shock_start_idx = int(tr['shock_start_idx'])
G0_shock = tr['dist_at_shock_start']
k_initial = float(tr['k_agg'][shock_start_idx])
q_initial = float(tr['q_agg'][shock_start_idx - 1])
tot_initial = float(tr['tot'][shock_start_idx - 1])
BG_shock = float(shock_b_agg[shock_start_idx - 1])
k_calibrated = k_ss     # == aggregates_calibrated.k
l_calibrated = 1.0       # == aggregates_calibrated.l
k_final = k_calibrated

# Leverage
k_const_hat = 1.0 / 16.0
k_const = k_const_hat * k_initial
phi_ac = 17.0

print(f'TT = {TT}, shock_start_idx = {shock_start_idx} (= MATLAB period {shock_start_idx + 1})')
print(f'k_initial   = {k_initial:.6f}, q_initial = {q_initial:.6f}, tot_initial = {tot_initial:.6f}')
print(f'BG_shock    = {BG_shock:.6f}  (households\' foreign-currency liabilities)')
print(f'k_final     = {k_final:.6f}, k_const = {k_const:.6f}, k_const_hat = {k_const_hat:.4f}')
print(f'new_shock_b_agg[0] = {new_shock_b_agg[0]:.6f}, [-1] = {new_shock_b_agg[-1]:.6f}')
print(f'G0_shock sum = {G0_shock.sum():.10f}')

TT = 200, shock_start_idx = 40 (= MATLAB period 41)
k_initial   = 29.751312, q_initial = 1.017149, tot_initial = 1.025849
BG_shock    = -6.281006  (households' foreign-currency liabilities)
k_final     = 29.715189, k_const = 1.859457, k_const_hat = 0.0625
new_shock_b_agg[0] = -6.281006, [-1] = -4.343250
G0_shock sum = 1.0000000000


## Helpers + parameterised `backsolve_egm` and `solve_transition_bis`

Same logic as notebook 10, but the contraction-specific initial conditions and `k_const_hat = 1/16` are passed in as arguments rather than module-level constants.

In [2]:
def basefun_vec(grid, x):
    n = grid.size
    ind1 = np.searchsorted(grid, x, side='right') - 1
    ind1 = np.clip(ind1, 0, n - 2)
    ind2 = ind1 + 1
    w2 = (x - grid[ind1]) / (grid[ind2] - grid[ind1])
    w2 = np.clip(w2, 0.0, 1.0)
    w1 = 1.0 - w2
    return ind1, ind2, w1, w2


def backsolve_egm(r_t, t_guess, y_agg, cpol, c_fine, prof_firm, Profit_i, G0,
                  *, k_init_arg, q_init_arg, tot_init_arg, BG_shock_arg,
                  k_const_hat_arg, k_const_arg, foreign_arg=True):
    sav_grid = np.tile(Agrid[:, None], (1, ns))
    exLw = np.tile(ex[None, :], (nA, 1))

    pos = Agrid_fine > 0
    net_a_rich = (G0[pos, :].sum(axis=1) * Agrid_fine[pos]).sum()
    if BG_shock_arg == 0.0:
        lev_rat_rich = 1.0
    else:
        lev_rat_rich = (1.0 - k_const_hat_arg) * k_init_arg * q_init_arg / max(net_a_rich, 1e-12)
    Kgrid_fine = np.maximum(lev_rat_rich * Agrid_fine + k_const_arg, 0.0)
    Bgrid_fine = -(Kgrid_fine - Agrid_fine)

    c_tran = np.full((nA, ns, TT), np.nan)
    c_tran_fine_out = np.full((nA_fine, ns, TT), np.nan)
    dec_tran = np.full((nA, ns, TT), np.nan)
    dec_tran_fine = np.full((nA_fine, ns, TT), np.nan)
    c_tran[:, :, TT - 1] = cpol
    c_tran_fine_out[:, :, TT - 1] = c_fine

    for t in range(TT - 2, -1, -1):
        r = r_t[t + 1]
        r_yesterday = r_t[t]
        tot_t = t_guess[t]
        y_t_agg = y_agg[t]
        prof_firms_t = prof_firm[t] + Profit_i[t]
        mult = 1.0 + tot_t ** (theta - 1.0) * omega / (1.0 - omega)
        c_constrained = np.maximum(
            (1.0 / mult) * (
                (1.0 + r_yesterday) * sav_grid
                + (epsilon_f - 1.0) / epsilon_f * exLw * y_t_agg * (1.0 - alpha) / ex_mean
                + prof_firms_t - bmin
            ),
            1e-5,
        )
        Emup1 = c_tran[:, :, t + 1] ** (-gamma)
        Emup = beta_calibrated * (1.0 + r) * (Emup1 @ piex.T)
        c_s = Emup ** (-1.0 / gamma)
        a_today = (
            c_s * mult + sav_grid
            - (epsilon_f - 1.0) / epsilon_f * exLw * y_t_agg * (1.0 - alpha) / ex_mean
            - prof_firms_t
        ) / (1.0 + r_yesterday)
        c_new = np.empty((nA, ns))
        c_new_fine = np.empty((nA_fine, ns))
        dec_temp_fine = np.empty((nA_fine, ns))
        for j in range(ns):
            xj = a_today[:, j]
            cj = c_s[:, j]
            threshold = a_today[Index_b_min, j]
            mask = Agrid > threshold
            interp = PchipInterpolator(xj, cj, extrapolate=True)
            c_new[:, j] = np.where(mask, interp(Agrid), c_constrained[:, j])
            c_new[:, j] = np.maximum(c_new[:, j], 1e-5)
            mask_fine = Agrid_fine > threshold
            interp_unc = PchipInterpolator(xj, cj, extrapolate=True)
            interp_con = PchipInterpolator(Agrid, c_constrained[:, j], extrapolate=True)
            c_new_fine[:, j] = np.where(
                mask_fine, interp_unc(Agrid_fine), interp_con(Agrid_fine)
            )
            dec_unc = (
                (1.0 + r_yesterday) * Agrid_fine
                + (epsilon_f - 1.0) / epsilon_f * ex[j] * y_t_agg * (1.0 - alpha) / ex_mean
                + prof_firms_t
                - c_new_fine[:, j] * mult
            )
            dec_temp_fine[:, j] = np.where(mask_fine, dec_unc, bmin)
        c_tran[:, :, t] = c_new
        dec_tran[:, :, t] = (
            (1.0 + r_yesterday) * sav_grid
            + (epsilon_f - 1.0) / epsilon_f * exLw * y_t_agg * (1.0 - alpha) / ex_mean
            + prof_firms_t - c_new * mult
        )
        if dec_temp_fine.min() < bmin - 1e-10:
            raise RuntimeError(f'dec_temp_fine < bmin at t={t}')
        c_tran_fine_out[:, :, t] = c_new_fine
        dec_tran_fine[:, :, t] = dec_temp_fine

    dist_tran = np.zeros((nA_fine, ns, TT))
    if foreign_arg:
        Bgrid_fine_new = Bgrid_fine / (t_guess[0] / tot_init_arg)
        A_grid_fine_new = Bgrid_fine_new + Kgrid_fine
        if bmin / t_guess[0] < Agrid_fine[0]:
            raise RuntimeError('positive mass would land outside Agrid_fine')
        ind1, ind2, w1, w2 = basefun_vec(Agrid_fine, A_grid_fine_new)
        G0_new = np.zeros_like(G0)
        for ai in range(nA_fine):
            for j in range(ns):
                G0_new[ind1[ai], j] += G0[ai, j] * w1[ai]
                G0_new[ind2[ai], j] += G0[ai, j] * w2[ai]
        if t_guess[0] > 1:
            G0_new = np.nan_to_num(G0_new, nan=0.0)
        if G0_new.sum() < 1.0 - 1e-6:
            raise RuntimeError('Revaluated G0 does not sum to one')
        dist_tran[:, :, 0] = G0_new
    else:
        dist_tran[:, :, 0] = G0

    a_prime_t = np.zeros(TT)
    c_t_out = np.zeros(TT)
    a_prime_t[0] = (dist_tran[:, :, 0] * dec_tran_fine[:, :, 0]).sum()
    c_t_out[0] = (dist_tran[:, :, 0] * c_tran_fine_out[:, :, 0]).sum()

    for t in range(1, TT - 1):
        prev = dist_tran[:, :, t - 1]
        ind1, ind2, w1, w2 = basefun_vec(Agrid_fine, dec_tran_fine[:, :, t - 1])
        new_dist = np.zeros((nA_fine, ns))
        for j in range(ns):
            mass = prev[:, j]
            contrib_low = mass * w1[:, j]
            contrib_hi = mass * w2[:, j]
            for jp in range(ns):
                np.add.at(new_dist[:, jp], ind1[:, j], contrib_low * piex[j, jp])
                np.add.at(new_dist[:, jp], ind2[:, j], contrib_hi * piex[j, jp])
        dist_tran[:, :, t] = new_dist
        a_prime_t[t] = (new_dist * dec_tran_fine[:, :, t]).sum()
        c_t_out[t] = (new_dist * c_tran_fine_out[:, :, t]).sum()
    return c_t_out, a_prime_t, dec_tran, dec_tran_fine, c_tran_fine_out, dist_tran[:, :, 0], dist_tran


def solve_transition_bis(XX, z_t_in, shock_b_agg_in, cpol, c_fine, G0,
                          *, k_init_arg, k_final_arg, q_init_arg, tot_init_arg,
                          BG_shock_arg, k_const_hat_arg, k_const_arg):
    k_agg = np.empty(TT)
    k_agg[0] = k_init_arg
    k_agg[1:] = XX[:TT - 1]
    lab_agg = XX[TT - 1:]
    i_agg = k_agg[1:] - (1.0 - delta) * k_agg[:-1]
    q_agg = np.empty(TT)
    q_agg[:TT - 1] = 1.0 + phi_ac * (i_agg - delta * k_agg[:TT - 1]) / k_agg[:TT - 1]
    q_agg[TT - 1] = 1.0
    y_agg = z_t_in * (k_agg ** alpha) * (lab_agg ** (1.0 - alpha))
    r_k = (epsilon_f - 1.0) / epsilon_f * z_t_in * alpha * (lab_agg / k_agg) ** (1.0 - alpha)
    w_r = (epsilon_f - 1.0) / epsilon_f * z_t_in * (1.0 - alpha) * (lab_agg / k_agg) ** (-alpha)
    q_lag = np.empty(TT); q_lag[0] = q_init_arg; q_lag[1:] = q_agg[:-1]
    r_t = (r_k + (1.0 - delta) * q_agg) / q_lag - 1.0
    a_prime = np.empty(TT)
    a_prime[:TT - 1] = shock_b_agg_in[:TT - 1] + q_agg[:TT - 1] * k_agg[1:]
    a_prime[TT - 1] = q_agg[TT - 1] * k_final_arg + shock_b_agg_in[TT - 1]
    a_t_agg = np.empty(TT)
    a_t_agg[0] = q_init_arg * k_agg[0] + BG_shock_arg
    a_t_agg[1:] = a_prime[:-1]
    Profits_i = np.empty(TT)
    Profits_i[:TT - 1] = (
        (q_agg[:TT - 1] - 1.0) * i_agg
        - (phi_ac / 2.0) * k_agg[:TT - 1] * ((i_agg - delta * k_agg[:TT - 1]) / k_agg[:TT - 1]) ** 2
    )
    Profits_i[TT - 1] = 0.0
    Profits_firms = (1.0 / epsilon_f) * y_agg
    Profits = Profits_i + Profits_firms

    t_guess = np.empty(TT)
    c_t_agg = np.empty(TT)
    net_c_arr = np.empty(TT)
    for t in range(TT):
        if t == 0:
            cap_resource = q_init_arg * k_agg[0]
            BG_term = BG_shock_arg
            tot_init_local = tot_init_arg
            def c_t_guess(x):
                return (1.0 / (1.0 + x ** (theta - 1.0) * omega / (1.0 - omega))) * (
                    (cap_resource + BG_term / (x / tot_init_local)) * (1.0 + r_t[0])
                    - a_prime[0] + w_r[0] * lab_agg[0] + Profits[0]
                )
        else:
            def c_t_guess(x, t=t):
                return (1.0 / (1.0 + x ** (theta - 1.0) * omega / (1.0 - omega))) * (
                    a_t_agg[t] * (1.0 + r_t[t]) - a_prime[t] + w_r[t] * lab_agg[t] + Profits[t]
                )
        if t < TT - 1:
            def net_cons(x, t=t):
                return (
                    y_agg[t] - i_agg[t] - x ** (-theta_star) * d_s
                    - (phi_ac / 2.0) * k_agg[t] * ((i_agg[t] - delta * k_agg[t]) / k_agg[t]) ** 2
                )
        else:
            def net_cons(x, t=t):
                return (
                    y_agg[t] - i_agg[t - 1] - x ** (-theta_star) * d_s
                    - (phi_ac / 2.0) * k_agg[t] * ((i_agg[t - 1] - delta * k_agg[t]) / k_agg[t]) ** 2
                )
        f = lambda x: c_t_guess(x) - net_cons(x)
        try:
            x0 = brentq(f, 0.5, 2.0, xtol=1e-12)
        except ValueError:
            x0 = float(fsolve(f, 0.8, full_output=False, xtol=1e-12)[0])
        t_guess[t] = x0
        c_t_agg[t] = c_t_guess(x0)
        net_c_arr[t] = net_cons(x0)
    tot = t_guess.copy()

    c_t, a_prime_t, dec_tran, dec_tran_fine, c_tran_fine, In_dis, dist_tran = backsolve_egm(
        r_t, tot, y_agg, cpol, c_fine, Profits_firms, Profits_i, G0,
        k_init_arg=k_init_arg, q_init_arg=q_init_arg, tot_init_arg=tot_init_arg,
        BG_shock_arg=BG_shock_arg, k_const_hat_arg=k_const_hat_arg, k_const_arg=k_const_arg,
    )
    c_t[TT - 1] = c_t[TT - 2]
    k_prime_t = (a_prime_t[:TT - 1] - shock_b_agg_in[:TT - 1]) / q_agg[:TT - 1]
    lab_prime = (1.0 - omega) * (epsilon_w - 1.0) / epsilon_w * w_r / (c_t * chi_dis)
    resid = np.empty(2 * TT - 1)
    resid[:TT - 1] = k_prime_t - k_agg[1:]
    resid[TT - 1:] = lab_prime - lab_agg
    return resid, dict(
        r_t=r_t, r_k=r_k, y_agg=y_agg, c_t=c_t, c_t_agg=c_t_agg, tot=tot,
        lab_agg=lab_agg, w_r=w_r, q_agg=q_agg, k_agg=k_agg,
        a_prime=a_prime, a_t_agg=a_t_agg,
        Profits=Profits, Profits_i=Profits_i, Profits_firms=Profits_firms,
        net_c=net_c_arr, In_dis=In_dis,
    )

## Build initial guess and check residual

From `MIT_transition.m` lines 134–141: $k(t) = k_\text{cal} + (k_\text{init} - k_\text{cal})\,0.8^t$, $l(t) = l_\text{cal}$ constant.

In [3]:
k_exp_guess = np.empty(TT)
l_exp_guess = np.empty(TT)
for t in range(TT):
    k_exp_guess[t] = k_calibrated + (k_initial - k_calibrated) * 0.8 ** (t + 1)
    l_exp_guess[t] = l_calibrated
k_l_exp_init = np.concatenate([k_exp_guess[:TT - 1], l_exp_guess])

tic = time.perf_counter()
resid_init, _ = solve_transition_bis(
    k_l_exp_init, z_t, new_shock_b_agg, cpol_calibrated, c_fine_calibrated, G0_shock,
    k_init_arg=k_initial, k_final_arg=k_final, q_init_arg=q_initial,
    tot_init_arg=tot_initial, BG_shock_arg=BG_shock,
    k_const_hat_arg=k_const_hat, k_const_arg=k_const,
)
elapsed_one = time.perf_counter() - tic
print(f'Initial residual at simple guess: max|r| = {np.max(np.abs(resid_init)):.3e}')
print(f'  max|r_k|   = {np.max(np.abs(resid_init[:TT-1])):.3e}')
print(f'  max|r_lab| = {np.max(np.abs(resid_init[TT-1:])):.3e}')
print(f'One residual evaluation took: {elapsed_one:.3f}s')

Initial residual at simple guess: max|r| = 1.550e+00
  max|r_k|   = 1.550e+00
  max|r_lab| = 9.864e-02
One residual evaluation took: 0.398s


## Solve via `scipy.optimize.fsolve` (timed)

In [4]:
def f_only(XX):
    r, _ = solve_transition_bis(
        XX, z_t, new_shock_b_agg, cpol_calibrated, c_fine_calibrated, G0_shock,
        k_init_arg=k_initial, k_final_arg=k_final, q_init_arg=q_initial,
        tot_init_arg=tot_initial, BG_shock_arg=BG_shock,
        k_const_hat_arg=k_const_hat, k_const_arg=k_const,
    )
    return r

tic_total = time.perf_counter()
XX_star, info, ier, msg = fsolve(
    f_only, k_l_exp_init, full_output=True, xtol=1e-9, maxfev=400 * 8,
)
elapsed_fsolve = time.perf_counter() - tic_total
print(f'fsolve: ier={ier}, nfev={info["nfev"]}, elapsed={elapsed_fsolve:.1f}s')
print(f'  message: {msg.strip()}')

resid_final, out = solve_transition_bis(
    XX_star, z_t, new_shock_b_agg, cpol_calibrated, c_fine_calibrated, G0_shock,
    k_init_arg=k_initial, k_final_arg=k_final, q_init_arg=q_initial,
    tot_init_arg=tot_initial, BG_shock_arg=BG_shock,
    k_const_hat_arg=k_const_hat, k_const_arg=k_const,
)
print(f'Final residual: max|r| = {np.max(np.abs(resid_final)):.3e}')

/var/folders/1f/0xqk1x955sv5c_32y0s7yzhw0000gn/T/ipykernel_90699/2457142826.py:199: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  x0 = float(fsolve(f, 0.8, full_output=False, xtol=1e-12)[0])


/var/folders/1f/0xqk1x955sv5c_32y0s7yzhw0000gn/T/ipykernel_90699/2457142826.py:199: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last five Jacobian evaluations.
  x0 = float(fsolve(f, 0.8, full_output=False, xtol=1e-12)[0])


fsolve: ier=1, nfev=414, elapsed=164.2s
  message: The solution converged.


Final residual: max|r| = 8.326e-11


## Quick numerical check

In [5]:
print('--- Impact period (t=0) ---')
print(f'  k_agg[0]   = {out["k_agg"][0]:.6f} (= k_initial)')
print(f'  k_agg[1]   = {out["k_agg"][1]:.6f}')
print(f'  lab_agg[0] = {out["lab_agg"][0]:.6f}')
print(f'  tot[0]     = {out["tot"][0]:.6f}  (terms of trade; <1 = depreciation)')
print(f'  c_t[0]     = {out["c_t"][0]:.6f}')
print(f'  r_t[0]     = {100*out["r_t"][0]:.3f}% per quarter (<0 = balance-sheet hit)')
print(f'  y_agg[0]   = {out["y_agg"][0]:.6f}')
print()
print('--- Long run (last 3 quarters) ---')
print(f'  k_agg[-3:]  = {out["k_agg"][-3:]}')
print(f'  c_t[-3:]    = {out["c_t"][-3:]}')
print(f'  tot[-3:]    = {out["tot"][-3:]}')

print()
print(f'=== Phase D timings (notebook 12) ===')
print(f'  one residual eval     = {elapsed_one:.3f}s')
print(f'  fsolve (n eval = {info["nfev"]}) = {elapsed_fsolve:.1f}s ({elapsed_fsolve/60:.2f} min)')

--- Impact period (t=0) ---
  k_agg[0]   = 29.751312 (= k_initial)
  k_agg[1]   = 29.672294
  lab_agg[0] = 1.065578
  tot[0]     = 0.909879  (terms of trade; <1 = depreciation)
  c_t[0]     = 1.336105
  r_t[0]     = -4.865% per quarter (<0 = balance-sheet hit)
  y_agg[0]   = 3.196974

--- Long run (last 3 quarters) ---
  k_agg[-3:]  = [29.70855009 29.70861509 29.70867378]
  c_t[-3:]    = [1.45319848 1.45320168 1.45320168]
  tot[-3:]    = [1.00002166 1.00002108 0.99877622]

=== Phase D timings (notebook 12) ===
  one residual eval     = 0.398s
  fsolve (n eval = 414) = 164.2s (2.74 min)


## Save outputs for notebook 14

In [6]:
out_path = OUTPUT_DIR / 'contraction_flex.npz'
np.savez(
    out_path,
    XX_star=XX_star, resid_final=resid_final,
    elapsed_seconds=elapsed_fsolve, nfev=info['nfev'],
    k_const_hat=k_const_hat, k_const=k_const,
    k_initial=k_initial, q_initial=q_initial, tot_initial=tot_initial,
    BG_shock=BG_shock, k_final=k_final,
    k_agg=out['k_agg'], lab_agg=out['lab_agg'], y_agg=out['y_agg'],
    tot=out['tot'], c_t=out['c_t'], c_t_agg=out['c_t_agg'],
    r_t=out['r_t'], r_k=out['r_k'], w_r=out['w_r'], q_agg=out['q_agg'],
    a_prime=out['a_prime'], a_t_agg=out['a_t_agg'],
    Profits=out['Profits'], Profits_i=out['Profits_i'],
    Profits_firms=out['Profits_firms'], net_c=out['net_c'],
    In_dis=out['In_dis'],
)
print(f'Saved: {out_path}')

Saved: /Users/siyingli/github/de-Ferra2020-kz/Code/Python/output/contraction_flex.npz
